---
# <div align="center"><font color='blue'>  </font></div>
# <div align="center"><font color='blue'> COSC 2779/2972 | Deep Learning </font></div>
## <div align="center"><font color='blue'> Week 3 Lab: Neural Network Optimisation</font></div>
---

## 🎯 Learning Objectives
By the end of this lab, you should be able to:
- Diagnose common training problems from loss curves
- Compare different optimisers (SGD, SGD + Momentum, Adam) and understand their trade-offs
- Investigate the effect of learning rate on training stability and convergence
- Experiment with activation functions and weight initialisation
- Apply regularisation techniques (Dropout, L2 / weight decay) to reduce overfitting
- Tune basic hyperparameters systematically using a validation set

**Prerequisites:** Week 2 PyTorch introduction (training loop, `nn.Module`, loss functions, `torch.optim`), Week 3: Pre-recorded Materials (on canvas).

---
## Setup

### Imports and Device Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### Dataset: MNIST
We use the MNIST handwritten digit dataset (28×28 grayscale images, 10 classes). This is a standard benchmark for testing optimisation and training procedures.

We will learn more on datasets and data loaders in a later lab, for now you can use them as below.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
])

full_train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

### Creating a Validation Set
> Recall from Week 2: the test set must remain untouched during model development. We split the training data into a training set and a validation set for hyperparameter tuning.

In [ ]:
# Use a smaller subset for faster iteration during the lab
torch.manual_seed(42)
indices = torch.randperm(len(full_train_dataset))
train_indices = indices[:10000]     # 10k for training
val_indices = indices[10000:12000]  # 2k for validation

train_dataset = Subset(full_train_dataset, train_indices)
val_dataset = Subset(full_train_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

---
### Model Definition
A simple Multi-Layer Perceptron (MLP) with one hidden layer. You will modify this throughout the lab.

👉 **Note:** The activation function is passed as a parameter so you can easily swap it in later exercises without rewriting the class.

In [ ]:
class MLP(nn.Module):
    def __init__(self, activation=nn.ReLU()):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            activation,
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.model(x)

---
### Training and Evaluation Functions
These functions implement the training loop you learned in Week 2. The key steps are:
1. Forward pass → 2. Compute loss → 3. Zero gradients → 4. Backward pass → 5. Optimiser step

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    """Train for one epoch and return average loss."""
    model.train()
    total_loss = 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        outputs = model(x)
        loss = criterion(outputs, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * y.size(0)
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion):
    """Evaluate model and return both average loss and accuracy."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            # Calculate loss
            loss = criterion(outputs, y)
            total_loss += loss.item() * y.size(0)

            # Calculate accuracy
            _, predicted = torch.max(outputs, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    avg_loss = total_loss / len(loader.dataset)
    accuracy = correct / total
    return avg_loss, accuracy

def run_experiment(model, optimizer, criterion, epochs=15, name="Model"):
    """Run a full training experiment and return history."""
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(epochs):
        # 1. Train for one epoch
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion)

        # 2. Evaluate on both sets
        _, train_acc = evaluate(model, train_loader, criterion)
        val_loss, val_acc = evaluate(model, val_loader, criterion)

        # 3. Save to history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  [{name}] Epoch {epoch+1}/{epochs} — "
                  f"Loss: {train_loss:.4f}, Val Acc: {val_acc:.4f}")

    return history

### Plotting Helper

In [ ]:
def plot_experiments(results,
                     show_train=True, show_val=True, show_accuracy=True):
    """
    Plot training/validation loss and accuracy for multiple experiments.

    Args:
        results: dict of {name: history_dict}
        show_train: whether to plot training curves
        show_val: whether to plot validation curves
        show_accuracy: whether to include the accuracy subplot
    """
    n_plots = 2 if show_accuracy else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(7 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]  # make iterable when only one subplot

    # --- Loss ---
    ax = axes[0]
    for name, hist in results.items():
        epochs = range(1, len(hist['train_loss']) + 1)
        if show_train:
            ax.plot(epochs, hist['train_loss'], label=f'{name} (Train)', marker='o', markersize=4)
        if show_val and 'val_loss' in hist:
            ax.plot(epochs, hist['val_loss'], label=f'{name} (Val)', marker='x', markersize=4, linestyle='--')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.7)

    # --- Accuracy ---
    if show_accuracy:
        ax = axes[1]
        for name, hist in results.items():
            epochs = range(1, len(hist.get('train_acc', hist['val_acc'])) + 1)
            if show_train and 'train_acc' in hist:
                ax.plot(epochs, hist['train_acc'], label=f'{name} (Train)', marker='o', markersize=4)
            if show_val and 'val_acc' in hist:
                ax.plot(epochs, hist['val_acc'], label=f'{name} (Val)', marker='x', markersize=4, linestyle='--')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend()
        ax.grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    plt.show()

---
## Part 1: Baseline — When Training Goes Wrong
Before we optimise anything, let's see what happens when things go wrong. The settings below are **intentionally poor**.

### Task
Train the model with these settings and observe the behaviour:

In [ ]:
torch.manual_seed(0)
model = MLP().to(device)
optimizer = optim.SGD(model.parameters(), lr=2.0)  # intentionally too high
criterion = nn.CrossEntropyLoss()

results = {}
results['SGD lr=2.0'] = run_experiment(model, optimizer, criterion, epochs=15, name='SGD lr=0.5')
plot_experiments(results)

### 🤔 Questions
1. Does the loss decrease smoothly, or is it unstable/diverging?
2. What is the final validation accuracy? Is it reasonable for MNIST?
3. Based on your knowledge of gradient descent, what is the underlying cause of this training failure? Explain the mathematical reasons for this behavior.
4. What specific visual indicators in your plotted learning curves point you toward this conclusion?
5. Change the learning rate from `lr=2.0` to `lr=0.01` and re-run the experiment. How does this adjustment change the training dynamics and the final accuracy?

👉 Note: When diagnosing optimiser-related issues, we primarily look at the training loss curve. This is because the optimiser directly minimises the training loss, so its behaviour (convergence speed, stability, divergence) is most clearly visible there. Validation metrics tell us about generalisation, which is influenced by many other factors beyond the optimiser (model capacity, regularisation, data quality, etc.).

*Double-click here to write your answers for Part 1...*

---
## Part 2: Comparing Optimisers
### Task
Train the same model architecture using three different optimisers, all with **the same learning rate** for a fair comparison. Plot all loss curves on the same graph.

In [ ]:
results = {}
criterion = nn.CrossEntropyLoss()
lr = 0.01

# --- SGD (no momentum) ---
torch.manual_seed(0)
model_sgd = MLP().to(device)
opt_sgd = optim.SGD(model_sgd.parameters(), lr=lr)
results['SGD'] = run_experiment(model_sgd, opt_sgd, criterion, epochs=15, name='SGD')

# --- SGD + Momentum ---
torch.manual_seed(0)
model_mom = MLP().to(device)
opt_mom = optim.SGD(model_mom.parameters(), lr=lr, momentum=0.9)
results['SGD+Momentum'] = run_experiment(model_mom, opt_mom, criterion, epochs=15, name='SGD+Momentum')

# --- Adam ---
torch.manual_seed(0)
model_adam = MLP().to(device)
opt_adam = optim.Adam(model_adam.parameters(), lr=lr)
results['Adam'] = run_experiment(model_adam, opt_adam, criterion, epochs=15, name='Adam')

# Plot comparison
plot_experiments(results, show_val=False, show_accuracy=False)

### 🤔 Questions
1. Which optimiser converges fastest?
2. Which produces the most stable (smooth) loss curve?
3. Based on what you learned in the lecture, *why* does Adam typically converge faster than vanilla SGD?
4. Are there any downsides to always using Adam?

*Double-click here to write your answers for Part 2...*

---
## Part 3: Learning Rate Exploration
### Task
Using **Adam** as the optimiser, investigate how the learning rate affects training. Train with each of the following learning rates and plot all results together.

In [ ]:
results = {}
criterion = nn.CrossEntropyLoss()

for lr in [1e-4, 1e-3, 1e-2, 1e-1]:
    torch.manual_seed(0)
    model = MLP().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    name = f'lr={lr}'
    results[name] = run_experiment(model, optimizer, criterion, epochs=15, name=name)

plot_experiments(results, show_val=False, show_accuracy=False)

### 🤔 Questions
1. Which learning rate is too small? What does its loss curve look like?
2. Which learning rate is too large? What happens?
3. Which learning rate gives the best performance?
4. 🤔 **Think:** If you only had a limited compute budget (say, 5 epochs), would your choice of learning rate change? Why?

*Double-click here to write your answers for Part 3...*

---
## Part 4: Activation Functions and Weight Initialisation
### Task A: Activation Functions
Modify the model to use different activation functions and compare their training behaviour.

In [ ]:
results = {}
criterion = nn.CrossEntropyLoss()

activations = {
    'ReLU': nn.ReLU(),
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
}

for name, act in activations.items():
    torch.manual_seed(0)
    model = MLP(activation=act).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    results[name] = run_experiment(model, optimizer, criterion, epochs=15, name=name)

plot_experiments(results, show_val=False, show_accuracy=False)

### Task B: Weight Initialisation
Compare default PyTorch initialisation with Xavier (Glorot) initialisation.

In [ ]:
def init_xavier(model):
    """Apply Xavier initialisation to all linear layers."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.xavier_uniform_(m.weight)
            nn.init.zeros_(m.bias)

results = {}
criterion = nn.CrossEntropyLoss()

# Default init
torch.manual_seed(0)
model_default = MLP(activation=nn.Sigmoid()).to(device)
optimizer = optim.SGD(model_default.parameters(), lr=0.01)
results['Sigmoid + Default Init'] = run_experiment(model_default, optimizer, criterion, epochs=15, name='Default Init')

# Xavier init
torch.manual_seed(0)
model_xavier = MLP(activation=nn.Sigmoid()).to(device)
init_xavier(model_xavier)
optimizer = optim.SGD(model_xavier.parameters(), lr=0.01)
results['Sigmoid + Xavier Init'] = run_experiment(model_xavier, optimizer, criterion, epochs=15, name='Xavier Init')

plot_experiments(results, show_val=False, show_accuracy=False)

### 🤔 Questions
1. Which activation function trains fastest and achieves the best accuracy?
2. Do you observe slower learning with Sigmoid compared to ReLU? Why might this happen? (Hint: think about the gradient plots from the Week 2 lecture.)
3. Does Xavier initialisation help with Sigmoid? Why?
4. Would you expect Xavier initialisation to make a big difference with ReLU? Try it and check.

*Double-click here to write your answers for Part 4...*

---
## Part 5: Regularisation [Home work]
<font color='red'>If time is limited, this part can be skipped. We will discuss regularisation again next week. Go to Part 6</font>

In the Week 3 lecture, you learned about regularisation techniques to combat overfitting. Let's put them into practice.

### Task A: L2 Regularisation (Weight Decay)
In PyTorch, L2 regularisation is implemented via the `weight_decay` parameter in the optimiser.

In [ ]:
results = {}
criterion = nn.CrossEntropyLoss()

for wd in [0.0, 1e-4, 1e-3, 1e-2]:
    torch.manual_seed(0)
    model = MLP().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=wd)
    name = f'wd={wd}'
    results[name] = run_experiment(model, optimizer, criterion, epochs=15, name=name)

plot_experiments(results, show_val=True)

### Task B: Dropout
Create a model variant with Dropout and compare it to the baseline.

In [ ]:
class MLPWithDropout(nn.Module):
    def __init__(self, dropout_rate=0.5):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.model(x)

results = {}
criterion = nn.CrossEntropyLoss()

for dr in [0.0, 0.2, 0.5, 0.8]:
    torch.manual_seed(0)
    model = MLPWithDropout(dropout_rate=dr).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    name = f'Dropout={dr}'
    results[name] = run_experiment(model, optimizer, criterion, epochs=15, name=name)

plot_experiments(results, show_val=True, show_accuracy=False)

👉 Note: More data is an excellent regulariser, often more effective than any explicit technique like Dropout or weight decay, provided the model has adequate capacity. Try increasing the dataset size to the full 50k training samples and observe the effect on the train/val gap yourself.

### 🤔 Questions
1. Does increasing weight decay always improve validation accuracy? What happens when it's too large?
2. What effect does Dropout have on the training loss curve compared to without Dropout? Why?
3. What Dropout rate gives the best validation accuracy?
4. 🤔 **Think:** Dropout is turned off during evaluation (`model.eval()`). Why is this important? What would happen if we forgot to call `model.eval()` before computing validation accuracy?

*Double-click here to write your answers for Part 5...*

---
## Part 6: Debugging Challenge (20 min)
### The Broken Model
The code below has **multiple problems**. Your task is to identify and fix them.

For this section, you can use the full dataset. Go to section on `Creating a Validation set` then change the first part of the code to:

```
# Use the full dataset
torch.manual_seed(42)
indices = torch.randperm(len(full_train_dataset))
train_indices = indices[:50000]     # 50k for training
val_indices = indices[50000:]  # Remaining 10k for validation
```

In [ ]:
class BrokenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.Sigmoid(),
            nn.Linear(256, 128),
            nn.Sigmoid(),
            nn.Linear(128, 64),
            nn.Sigmoid(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.model(x)

torch.manual_seed(0)
broken_model = BrokenMLP().to(device)
optimizer = optim.SGD(broken_model.parameters(), lr=10.0)
criterion = nn.CrossEntropyLoss()

broken_results = {}
broken_results['Broken'] = run_experiment(broken_model, optimizer, criterion, epochs=10, name='Broken')

plot_experiments(broken_results)

### Instructions
1. Run the broken code and observe the training behaviour.
2. Diagnose the problem(s) — there are at least **two** issues.
3. Fix the issues by modifying the learning rate, optimiser, activation function, and/or initialisation. **Do NOT change the model size or dataset.**
4. Show a "before and after" comparison plot.

Write a short paragraph (5–10 sentences) explaining:
- What the problems were
- How you identified them (what evidence from the loss curve or accuracy?)
- What changes you made to fix them
- What you learned about the relationship between activation functions, learning rates, and optimiser choice

*Double-click here to write your debugging analysis...*

---
## Part 7: Reflection
Answer the following questions in your own words:
1. What does a "healthy" loss curve look like? What does a "bad" one look like?
2. What is the **first** thing you would check if a model fails to train?
3. When would you choose Adam over SGD? Are there cases where SGD might be preferable?
4. How does the learning rate interact with the choice of optimiser?
5. What is the purpose of regularisation, and how do Dropout and L2 (weight decay) achieve it differently?

*Double-click here to write your reflection answers...*

---
## Bonus (Optional)
If you finish early, try one or more of the following:
- **Learning rate scheduler:** Use `torch.optim.lr_scheduler.StepLR` or `CosineAnnealingLR` to decay the learning rate during training. Does it improve final performance?
- **Batch size effect:** Try batch sizes of 16, 64, 256, and 1024. How does batch size affect training speed and final accuracy?
- **Gradient monitoring:** Track the average gradient norm per epoch. How does it change with different activation functions?

## 🔑 Key Takeaway
Optimisation is not just about picking an algorithm — it is about understanding *how* and *why* training succeeds or fails. The ability to read a loss curve, diagnose a problem, and systematically fix it is one of the most valuable skills in deep learning.